# Protein Secondary Structure Prediction

### Problem Statement
Proteins are fundamental biological molecules whose functions are determined by their three-dimensional structures. Protein secondary structure describes local structural patterns such as helices, strands, and coils, which serve as building blocks for the full protein fold.

The objective of this project is to **predict the secondary structure of protein sequences directly from their amino acid sequences**. This is formulated as a **sequence-to-sequence prediction task**, where each amino acid in the input sequence is assigned a corresponding structural label.

Two annotation schemes are considered:
- **Q8 (Eight-state)**: A fine-grained representation of secondary structure states.
- **Q3 (Three-state)**: A simplified representation derived from Q8, grouping related structural states.


### Project Overview
In this project, deep learning-based sequence models are developed to predict protein secondary structure at the residue level. The workflow includes:

- Loading and exploring the protein sequence dataset
- Analyzing sequence length and label distributions
- Generating vocabulary, padding and masking
- Training bidirectional recurrent neural network models
- Evaluating performance using token-level F1 scores
- Generating predictions for unseen test sequences

Two bidirectional models are implemented:
- Bidirectional RNN
- Bidirectional LSTM

The final predictions are formatted according to the competition requirements and submitted for leaderboard evaluation.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sep-25-dl-gen-ai-nppe-2/sample_submission.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv
/kaggle/input/protein_ssp/pytorch/bilstm_v1/3/bilstm_model1.pth
/kaggle/input/protein_ssp/pytorch/bilstm_v1/2/bilstm_model1.pth
/kaggle/input/protein_ssp/pytorch/bilstm_v1/1/bilstm_model1.pth
/kaggle/input/protein_ssp/pytorch/birnn_v1/3/birnn_model1.pth
/kaggle/input/protein_ssp/pytorch/birnn_v1/5/birnn_model1.pth
/kaggle/input/protein_ssp/pytorch/birnn_v1/1/birnn_model1.pth


# 1. Setup & Initialization


###  Steps done:

1. **Install Trackio**   
   TrackIO is used for tracking metrics such as loss, and F1_score for each model during training.

2. **Hugging Face Token Setup**  
   Retrieve the stored Hugging Face access token using Kaggle Secrets and set it as an environment variable.  
   This ensures secure access to your TrackIO project hosted on Hugging Face Spaces.

3. **Importing Libraries**  
   All the required libraries are imported.

4. **Device Configuration**  
   For selecting device type (CPU or GPU).


In [2]:
# Installing trackio
!pip install trackio -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.0/988.0 kB 16.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 113.4 MB/s eta 0:00:0000:0100:01


In [3]:
# Adding huggingface access token to environment variable

from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ['HF_TOKEN']  = user_secrets.get_secret("hf_token")

print("HF token loaded successfully")

HF token loaded successfully


In [4]:
# Importing Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Dataset
from sklearn.metrics import f1_score, accuracy_score
import trackio

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# 2. Data Loading and Exploratory Data Analysis (EDA)

### 1. Data Loading
The dataset is loaded from CSV files using pandas.

- **train.csv** contains protein sequences along with their secondary structure annotations in Q8 and Q3 formats.
- **test.csv** contains only protein sequences for which secondary structure predictions must be generated.


### 2. Dataset Structure
The dataset is examined to understand its size and column structure.

- `train_df` : (7262,4)
- `test_df` : (1816,2)


### 3. Sequence Length Analysis
Protein sequences vary significantly in length.

- Sequence length statistics help identify variability across samples.
- This analysis motivates the need for padding and masking during model training.


### 4. Amino Acid Distribution
The distribution of amino acids is analyzed across all training sequences.

- Common amino acids appear frequently.
- Rare and non-standard residues occur infrequently.

This confirms the suitability of a fixed amino acid vocabulary.


### 5. Secondary Structure Label Distribution

- **Q8 labels** are highly imbalanced, with some structural states appearing very rarely.
- **Q3 labels**, derived from Q8, are more balanced and therefore easier to predict.


### 6. Label Consistency Check
A consistency check confirms that:

- Each protein sequence has corresponding Q8 and Q3 labels.
- Label lengths exactly match the input sequence length.

This ensures proper alignment between inputs and targets.


In [6]:
# Loading the csv files
train_df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv")
test_df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv")

# Getting shape of train and test sets
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Getting first 5 rows of train csv
train_df.head()

Train shape: (7262, 4)
Test shape: (1816, 2)


,id,seq,sst8,sst3
0,0,GVGLEGGVQLSPARTRGPEFAAPEQAG,CCCCCCCCSCCCCCCGGGCCCCCCCCC,CCCCCCCCCCCCCCCHHHCCCCCCCCC
1,1,NHGKVKIEHTKWNVEYKVTYNRNVFANHIRSGELASNGYHTTRRTA...,CEEEEEECCTTTEEEEEEEEEEEEEEEEEEEEECSCCSSSCCCCEE...,CEEEEEECCCCCEEEEEEEEEEEEEEEEEEEEECCCCCCCCCCCEE...
2,2,EMRKMLADWKGLSKSDGMLSSEGRTKALWLGEANFSYVPKLDPRAS...,CTHHHHHHHHHSGGGCCCCCCCCCCCCEEECSSCEEEEEEETTCGG...,CCHHHHHHHHHCHHHCCCCCCCCCCCCEEECCCCEEEEEEECCCHH...
3,3,QDNKNGWQIRSDDVWGPDTKDSIQTVEGTRDNVVVYKGPSGYVTAP...,CCCCCCEECTTCCBCTTCTTCCEEEEBCTTTEEEEEETTEEEEEEE...,CCCCCCEECCCCCECCCCCCCCEEEEECCCCEEEEEECCEEEEEEE...
4,4,VPNSRDGGGGNHWNVEFGQLIALIGAAICGVIGGALGGFTAAGSCG...,CEEEEEECCCHHHHHHTHHHHHHHHHHHHHHHHSCCTTCCCCSSCC...,CEEEEEECCCHHHHHHCHHHHHHHHHHHHHHHHCCCCCCCCCCCCC...


In [7]:
# Compute sequence lengths
seq_lengths = train_df["seq"].str.len()

# Print summary statistics
print(seq_lengths.describe())

count    7262.000000
mean      242.897411
std       157.307867
min        20.000000
25%       131.000000
50%       207.000000
75%       324.000000
max      1632.000000
Name: seq, dtype: float64


In [8]:
# Check if any sequence-label length mismatches exist
mismatch_q8 = (train_df["seq"].str.len() != train_df["sst8"].str.len()).sum()
mismatch_q3 = (train_df["seq"].str.len() != train_df["sst3"].str.len()).sum()

print("Q8 mismatches:", mismatch_q8)
print("Q3 mismatches:", mismatch_q3)

Q8 mismatches: 0
Q3 mismatches: 0


In [9]:
# Check frequency distribution of amino acids and secondary structure labels
from collections import Counter

aa_counts = Counter("".join(train_df["seq"]))
print(aa_counts)
print()
q8_counts = Counter("".join(train_df["sst8"]))
print(q8_counts)
print()
q3_counts = Counter("".join(train_df["sst3"]))
print(q3_counts)

Counter({'L': 160451, 'A': 145476, 'G': 129152, 'V': 119941, 'E': 117645, 'S': 110797, 'D': 104756, 'K': 99311, 'I': 97404, 'T': 95817, 'R': 89448, 'P': 81562, 'N': 76164, 'F': 70554, 'Q': 66225, 'Y': 61635, 'H': 51041, 'M': 40324, 'W': 25114, 'C': 20968, '*': 136})

Counter({'H': 556686, 'C': 427095, 'E': 370480, 'T': 187629, 'S': 136515, 'G': 66580, 'B': 18612, 'I': 324})

Counter({'C': 751239, 'H': 623590, 'E': 389092})


# 3. Data Preprocessing

### 1. Vocabulary Definition
A fixed vocabulary is constructed for amino acid sequences.

- Standard amino acids are included.
- Non-standard residues are represented using a special masking symbol.
- A `<PAD>` token is used to handle variable-length sequences.


### 2. Sequence Encoding
Protein sequences are converted into numerical form by mapping each amino acid to an integer index.

- This allows sequences to be processed by embedding layers.
- The same encoding scheme is applied consistently across training and test data.


### 3. Label Encoding
Secondary structure labels are encoded numerically.

- Q8 labels are mapped to eight integer classes.
- Q3 labels are mapped to three integer classes.

This enables the use of standard classification loss functions.


### 4. Padding and Masking
Since protein sequences vary in length:

- Sequences are padded to the maximum length within each batch.
- Padding positions are ignored during loss and metric computation using binary masks.


### 5. Train–Validation Split
The training dataset is split into training and validation subsets.

- The validation set is used to monitor model performance.
- A fixed random seed ensures reproducibility.


In [10]:
# Amino acid vocabulary (including masked *)
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY*")

# Map amino acid → index
aa2idx = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa2idx["<PAD>"] = 0

# Reverse map (index → amino acid)
idx2aa = {i: aa for aa, i in aa2idx.items()}

print("Amino acid to index mapping:")
print(aa2idx)
print()


# Q8 labels
Q8_LABELS = list("HGIEBTSC")

q8_2idx = {label: i for i, label in enumerate(Q8_LABELS)}
idx2q8 = {i: label for label, i in q8_2idx.items()}

print("Q8 label mapping:")
print(q8_2idx)
print()


# Q3 labels
Q3_LABELS = list("HEC")

q3_2idx = {label: i for i, label in enumerate(Q3_LABELS)}
idx2q3 = {i: label for label, i in q3_2idx.items()}

print("Q3 label mapping:")
print(q3_2idx)

Amino acid to index mapping:
{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, '*': 21, '<PAD>': 0}

Q8 label mapping:
{'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}

Q3 label mapping:
{'H': 0, 'E': 1, 'C': 2}


In [11]:
# Custom PyTorch dataset that converts protein sequences and their Q8 and Q3 secondary structure labels into numerical tensor representations

class ProteinDataset(Dataset):
    def __init__(self, dataframe, aa2idx, q8_2idx, q3_2idx):
        self.df = dataframe
        self.aa2idx = aa2idx
        self.q8_2idx = q8_2idx
        self.q3_2idx = q3_2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        seq  = self.df.loc[idx, "seq"]
        sst8 = self.df.loc[idx, "sst8"]
        sst3 = self.df.loc[idx, "sst3"]

        enc_seq = torch.tensor(
            [self.aa2idx[c] for c in seq],
            dtype=torch.long
        )
        enc_q8 = torch.tensor(
            [self.q8_2idx[c] for c in sst8],
            dtype=torch.long
        )
        enc_q3 = torch.tensor(
            [self.q3_2idx[c] for c in sst3],
            dtype=torch.long
        )

        return enc_seq, enc_q8, enc_q3


# Padding and Masking
def collate_fn(batch):
    sequences, q8_labels, q3_labels = zip(*batch)

    lengths = [len(seq) for seq in sequences]
    max_len = max(lengths)

    PAD_IDX = 0
    IGNORE_INDEX = -100

    padded_seqs = []
    padded_q8 = []
    padded_q3 = []
    masks = []

    for seq, q8, q3 in zip(sequences, q8_labels, q3_labels):
        pad_len = max_len - len(seq)

        padded_seqs.append(
            torch.cat([seq, torch.full((pad_len,), PAD_IDX)])
        )

        padded_q8.append(
            torch.cat([q8, torch.full((pad_len,), IGNORE_INDEX)])
        )
        padded_q3.append(
            torch.cat([q3, torch.full((pad_len,), IGNORE_INDEX)])
        )

        masks.append(
            torch.cat([torch.ones(len(seq)), torch.zeros(pad_len)])
        )

    return (
        torch.stack(padded_seqs),
        torch.stack(padded_q8),
        torch.stack(padded_q3),
        torch.stack(masks)
    )


In [12]:
# Create full dataset

full_dataset = ProteinDataset(train_df, aa2idx, q8_2idx, q3_2idx)
print("Total samples:", len(full_dataset))

Total samples: 7262


In [13]:
# Train / validation split

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 6535
Validation samples: 727


In [14]:
# Create DataLoaders to batch variable-length protein sequences for train and val sets

BATCH_SIZE = 10

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 654
Validation batches: 73


In [15]:
# Custom Dataset for test data that encodes protein sequences into integer tensors without labels

class ProteinTestDataset(Dataset):
    def __init__(self, dataframe, aa2idx):
        self.df = dataframe
        self.aa2idx = aa2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        seq = self.df.loc[idx, "seq"]
        enc_seq = torch.tensor(
            [self.aa2idx[c] for c in seq],
            dtype=torch.long
        )
        return enc_seq


#Padding and Masking of test dataset
def collate_fn_test(batch):
    sequences = batch
    lengths = [len(seq) for seq in sequences]
    max_len = max(lengths)

    PAD_IDX = 0

    padded_seqs = []
    masks = []

    for seq in sequences:
        pad_len = max_len - len(seq)

        padded_seqs.append(
            torch.cat([seq, torch.full((pad_len,), PAD_IDX)])
        )

        masks.append(
            torch.cat([torch.ones(len(seq)), torch.zeros(pad_len)])
        )

    return torch.stack(padded_seqs), torch.stack(masks)


In [16]:
# Test data loader

test_dataset = ProteinTestDataset(test_df, aa2idx)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_test
)

print("Test batches:", len(test_loader))

Test batches: 182


# 4. Models

Two bidirectional sequence models are implemented to predict protein secondary structure at the residue level. Both models perform token-level classification for **Q8** and **Q3** secondary structure schemes.


### 1. Bidirectional RNN Model

The Bidirectional RNN serves as a baseline sequence model.

**Architecture Overview:**
- An embedding layer converts amino acid indices into dense vector representations.
- A bidirectional vanilla RNN processes the sequence in both forward and backward directions.
- Layer normalization is applied to stabilize training.
- Dropout is used for regularization.
- Two separate fully connected output heads are used to predict Q8 and Q3 labels for each residue.

**Key Characteristics:**
- Captures contextual information from both sequence directions.
- Simpler architecture with limited ability to model long-range dependencies.
- Used as a baseline to compare against gated recurrent models.



### 2. Bidirectional LSTM Model

The Bidirectional LSTM improves upon the vanilla RNN by incorporating gating mechanisms.

**Architecture Overview:**
- Uses the same embedding layer as the RNN model.
- Replaces the vanilla RNN with a bidirectional LSTM to mitigate vanishing gradient issues.
- Layer normalization and dropout are applied to improve stability and generalization.
- Two task-specific output heads predict Q8 and Q3 labels independently.

**Key Characteristics:**
- Better modeling of long-range dependencies in protein sequences.
- More expressive than a vanilla RNN due to input, forget, and output gates.
- Expected to achieve higher performance, especially for Q8 prediction.



In [17]:
class BiRNNModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_layers,
        q8_classes=8,
        q3_classes=3,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity="tanh",
            batch_first=True,
            bidirectional=True
        )

        self.layer_norm = nn.LayerNorm(hidden_dim * 2)

        self.dropout = nn.Dropout(dropout)

        self.fc_q8 = nn.Linear(hidden_dim * 2, q8_classes)
        self.fc_q3 = nn.Linear(hidden_dim * 2, q3_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        out, _ = self.rnn(x)

        out = self.layer_norm(out)
        out = self.dropout(out)

        q8_logits = self.fc_q8(out)
        q3_logits = self.fc_q3(out)

        return q8_logits, q3_logits


In [20]:
# class BiLSTMModel(nn.Module):
#     def __init__(
#         self,
#         vocab_size,
#         embed_dim,
#         hidden_dim,
#         num_layers,
#         q8_classes=8,
#         q3_classes=3,
#         dropout=0.3
#     ):
#         super().__init__()

#         self.embedding = nn.Embedding(
#             vocab_size,
#             embed_dim,
#             padding_idx=0
#         )

#         self.lstm = nn.LSTM(
#             input_size=embed_dim,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True,
#             dropout=dropout if num_layers > 1 else 0.0
#         )

#         self.layer_norm = nn.LayerNorm(hidden_dim * 2)

#         self.dropout = nn.Dropout(dropout)

#         self.fc_q8 = nn.Linear(hidden_dim * 2, q8_classes)
#         self.fc_q3 = nn.Linear(hidden_dim * 2, q3_classes)

#     def forward(self, x):
#         x = self.embedding(x)
#         x = self.dropout(x)

#         out, _ = self.lstm(x)

#         out = self.layer_norm(out)
#         out = self.dropout(out)

#         q8_logits = self.fc_q8(out)
#         q3_logits = self.fc_q3(out)

#         return q8_logits, q3_logits


In [18]:
# Model hyperparameters
VOCAB_SIZE = len(aa2idx)
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1

model = BiRNNModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
)

model = model.to(device)


# Loss functions
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-100)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-100)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# 5. Model Training

### 1. Training Setup
Both models were trained using the same training configuration.

- Optimizer: Adam  
- Loss Function: Cross-Entropy Loss (masked for padding tokens)  
- Evaluation Metric: Token-level macro F1 score for Q8 and Q3  
- Batch Size: Fixed across models  
- Number of Epochs: 25  
- Gradient Clipping was applied to stabilize training  

Padding tokens were ignored during loss and metric computation using binary masks.



### 2. Training Logs Comparison

| Model | Best Train F1 (Q8) | Best Val F1 (Q8) | Best Train F1 (Q3) | Best Val F1 (Q3) | Epochs |
|------|------------------|------------------|------------------|------------------|--------|
| Bi-RNN | ~0.29 | ~0.28 | ~0.72 | ~0.69 | 25 |
| Bi-LSTM | ~0.31 | ~0.30 | ~0.71 | ~0.68 | 25 |



### 3. Observations
- Both models converged within the first 15–20 epochs.
- The Bidirectional LSTM showed slightly improved stability and generalization.
- Q3 prediction consistently outperformed Q8 due to reduced class complexity.
- Performance plateaued after convergence, indicating model capacity limits.


In [43]:
# # Computes macro F1 score while ignoring padded tokens using a mask

# def masked_f1(logits, labels, mask, num_classes):
#     preds = torch.argmax(logits, dim=-1)
#     preds = preds[mask == 1].cpu().numpy()
#     labels = labels[mask == 1].cpu().numpy()

#     return f1_score(
#         labels,
#         preds,
#         average="macro",
#         labels=list(range(num_classes)),
#         zero_division=0
#     )

In [24]:
# # Trackio initialization for Bi-RNN Model

# trackio.init(
#     project="25-t3-nppe2",
#     space_id="Ehsaas-Tiwari/dlgenai-nppe",
#     name="birnn_model1",
#     group="birnn"
# )

* Trackio project initialized: 25-t3-nppe2
* Trackio metrics will be synced to Hugging Face Dataset: Ehsaas-Tiwari/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/Ehsaas-Tiwari/dlgenai-nppe
* View dashboard by going to: https://Ehsaas-Tiwari-dlgenai-nppe.hf.space/


* Created new run: birnn_model1


In [44]:
# # Trackio initialization for Bi-LSTM Model

# trackio.init(
#     project="25-t3-nppe2",
#     space_id="Ehsaas-Tiwari/dlgenai-nppe",
#     name="bilstm_model1",
#     group="bilstm"
# )

* Created new run: bilstm_model1


In [19]:
# # Training Loop

# EPOCHS=25

# for epoch in range(EPOCHS):

#     # TRAINING
#     model.train()
#     train_loss = 0.0
#     train_f1_q8 = 0.0
#     train_f1_q3 = 0.0
#     num_batches = 0

#     for seqs, q8_labels, q3_labels, masks in train_loader:
#         seqs = seqs.to(device)
#         q8_labels = q8_labels.to(device)
#         q3_labels = q3_labels.to(device)
#         masks = masks.to(device)

#         optimizer.zero_grad()

#         q8_logits, q3_logits = model(seqs)

#         loss_q8 = criterion_q8(
#             q8_logits.view(-1, 8),
#             q8_labels.view(-1)
#         )

#         loss_q3 = criterion_q3(
#             q3_logits.view(-1, 3),
#             q3_labels.view(-1)
#         )

#         loss = loss_q8 + loss_q3
#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

#         optimizer.step()

#         train_loss += loss.item()
#         train_f1_q8 += masked_f1(q8_logits, q8_labels, masks, num_classes=8)
#         train_f1_q3 += masked_f1(q3_logits, q3_labels, masks, num_classes=3)

#         num_batches += 1

#     train_loss /= num_batches
#     train_f1_q8 /= num_batches
#     train_f1_q3 /= num_batches

#     # VALIDATION
#     model.eval()
#     val_loss = 0.0
#     val_f1_q8 = 0.0
#     val_f1_q3 = 0.0
#     num_batches = 0

#     with torch.no_grad():
#         for seqs, q8_labels, q3_labels, masks in val_loader:
#             seqs = seqs.to(device)
#             q8_labels = q8_labels.to(device)
#             q3_labels = q3_labels.to(device)
#             masks = masks.to(device)

#             q8_logits, q3_logits = model(seqs)

#             loss_q8 = criterion_q8(
#                 q8_logits.view(-1, 8),
#                 q8_labels.view(-1)
#             )

#             loss_q3 = criterion_q3(
#                 q3_logits.view(-1, 3),
#                 q3_labels.view(-1)
#             )

#             loss = loss_q8 + loss_q3

#             val_loss += loss.item()
#             val_f1_q8 += masked_f1(q8_logits, q8_labels, masks, num_classes=8)
#             val_f1_q3 += masked_f1(q3_logits, q3_labels, masks, num_classes=3)

#             num_batches += 1

#     val_loss /= num_batches
#     val_f1_q8 /= num_batches
#     val_f1_q3 /= num_batches

#     trackio.log({
#         "epoch": epoch + 1,
#         "train_loss": train_loss,
#         "val_loss": val_loss,
#         "train_f1_q8": train_f1_q8,
#         "val_f1_q8": val_f1_q8,
#         "train_f1_q3": train_f1_q3,
#         "val_f1_q3": val_f1_q3
#     })

#     print(
#         f"Epoch {epoch+1}/{EPOCHS} | "
#         f"Train Loss: {train_loss:.4f}, "
#         f"Train F1 Q8: {train_f1_q8:.4f}, "
#         f"Train F1 Q3: {train_f1_q3:.4f} | "
#         f"Val Loss: {val_loss:.4f}, "
#         f"Val F1 Q8: {val_f1_q8:.4f}, "
#         f"Val F1 Q3: {val_f1_q3:.4f}"
#     )


In [46]:
# trackio.finish()

* Run finished. Uploading logs to Trackio (please wait...)


In [19]:
# Saving the model
MODEL_PATH = "birnn_model1.pth"
torch.save(model.state_dict(), MODEL_PATH)
print("Model saved at:", MODEL_PATH)

Model saved at: birnn_model1.pth


# 6. Inference

KaggleHub was used to **store and version trained models** securely, enabling easy reusability without retraining during inference.

The **saved model was downloaded** from KaggleHub in the inference phase to generate predictions and create the final `submission.csv`.


In [20]:
# For RNN Model

import kagglehub

KAGGLE_USERNAME = "ehsaastiwari"
MODEL = "protein_ssp"
FRAMEWORK = "pytorch"
VARIATION = "birnn_v1"

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

model_path = "/kaggle/working/birnn_model1.pth"

kagglehub.model_upload(
    handle,
    model_path,
    version_notes="Final BiRNN Model"
)


Uploading Model https://www.kaggle.com/models/ehsaastiwari/protein_ssp/pytorch/birnn_v1 ...
Starting upload for file /kaggle/working/birnn_model1.pth


Uploading: 100%|██████████| 832k/832k [00:00<00:00, 1.96MB/s]

Upload successful: /kaggle/working/birnn_model1.pth (813KB)


Your model instance version has been created.
Files are being processed...
See at: https://www.kaggle.com/models/ehsaastiwari/protein_ssp/pytorch/birnn_v1


In [ ]:
# # For LSTM Model

# import kagglehub

# KAGGLE_USERNAME = "ehsaastiwari"
# MODEL = "protein_ssp"
# FRAMEWORK = "pytorch"
# VARIATION = "bilstm_v1"

# handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

# model_path = "/kaggle/working/bilstm_model1.pth"

# kagglehub.model_upload(
#     handle,
#     model_path,
#     version_notes="Final BiLSTM Model"
# )

In [21]:
# Download the model from KaggleHub
model_dir = kagglehub.model_download(handle)

In [22]:
# Loading the trained model

model_path = "/kaggle/input/protein_ssp/pytorch/birnn_v1/1/birnn_model1.pth"
model = BiRNNModel(
    vocab_size=len(aa2idx),
    embed_dim=128,
    hidden_dim=256,
    num_layers=1
).to(device)

model.load_state_dict(torch.load(model_path))
model.eval()

BiRNNModel(
  (embedding): Embedding(22, 128, padding_idx=0)
  (rnn): RNN(128, 256, batch_first=True, bidirectional=True)
  (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc_q8): Linear(in_features=512, out_features=8, bias=True)
  (fc_q3): Linear(in_features=512, out_features=3, bias=True)
)

# 7. Predictions

In [23]:
# Making Predictions on test dataset

all_q8_preds = []
all_q3_preds = []

with torch.no_grad():
    for seqs, masks in test_loader:
        seqs = seqs.to(device)

        # Forward pass
        q8_logits, q3_logits = model(seqs)

        # Class predictions
        q8_preds = torch.argmax(q8_logits, dim=-1).cpu()
        q3_preds = torch.argmax(q3_logits, dim=-1).cpu()

        # Remove padding using mask
        for i in range(seqs.size(0)):
            length = int(masks[i].sum().item())

            all_q8_preds.append(q8_preds[i][:length].tolist())
            all_q3_preds.append(q3_preds[i][:length].tolist())


pred_sst8 = ["".join(idx2q8[i] for i in seq) for seq in all_q8_preds]
pred_sst3 = ["".join(idx2q3[i] for i in seq) for seq in all_q3_preds]

# 8. Final Submission

In [24]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "sst8": pred_sst8,
    "sst3": pred_sst3
})
submission.to_csv("submission.csv", index=False)

# Conclusion

In this project, protein secondary structure prediction was formulated as a sequence-to-sequence learning problem using deep recurrent neural networks. Two bidirectional models were implemented and evaluated: a vanilla Bidirectional RNN and a Bidirectional LSTM.

- The **Bidirectional RNN** served as a strong baseline and achieved reasonable performance, demonstrating the feasibility of sequence modeling for this task reaching a public score more than 0.4.
- The **Bidirectional LSTM** showed modest but consistent improvements, particularly in capturing longer-range dependencies within protein sequences, but failed to out-perform Bi-RNN model on test set, reahcing a public score of only 0.13.

Across both models, prediction performance on Q3 labels was consistently higher than Q8 labels due to reduced class complexity and lower class imbalance. The results highlight the challenges associated with fine-grained Q8 prediction using simple recurrent architectures.

Overall, the experiments demonstrate that bidirectional recurrent models can effectively learn structural patterns from amino acid sequences. However, further improvements would likely require more advanced architectures, such as attention mechanisms, convolutional-recurrent hybrids, or the incorporation of evolutionary features.
